<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c06-header.png" alt="Nextia Learning · Deep Learning and Transformers Explained" width="100%">

# Transformer blocks

**[Transformer blocks](https://learning.nextia-ai.com/courses/deep-learning/m06/transformer-blocks/)** · Deep Learning and Transformers Explained · Module 6, lesson 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** build a transformer block from its four parts, check its shapes, and train the tiny transformer on Larkfield's tickets.

| | |
|---|---|
| **Time** | About 40 minutes. The training cell takes one to three minutes. |
| **Needs** | A browser and a free Colab or Kaggle account. The setup cells download the tickets (about 2 MB) and the course's `ticketnet.py`. In Kaggle, turn on Internet in the notebook settings. |
| **You should know** | One attention head, from [Attention step by step](https://learning.nextia-ai.com/courses/deep-learning/m05/attention-step-by-step/); the training loop, from [The training loop](https://learning.nextia-ai.com/courses/deep-learning/m03/the-training-loop/). |
| **Tested** | 2026-10-08, Python 3.12 with PyTorch 2.14.1 (CPU), in Jupyter on macOS, from a fresh start (*Restart and run all*); Colab and Kaggle have PyTorch preinstalled |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m06/transformer-blocks/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/usmanahmed99/nextia-learning-labs/blob/main/C06/M06-L01-transformer-blocks/transformer-blocks-solution.ipynb).

## Setup: check PyTorch

Run the next cell. It uses the PyTorch that is already installed (Colab and Kaggle have it) or installs the CPU version if it is missing; that can take two minutes the first time. You should see a line that starts with `PyTorch 2.` and the device `cpu`. The course uses the CPU only, so a graphics card is not needed. 

In [ ]:
try:
    import torch
except ImportError:  # not installed: install the CPU version
    %pip install -q --disable-pip-version-check torch --index-url https://download.pytorch.org/whl/cpu
    import torch
import platform
torch.manual_seed(0)
print("PyTorch", torch.__version__, "| Python", platform.python_version(), "| device cpu")

## Setup: get the tickets and the course's code

Run the next cell. It downloads Larkfield's ticket texts (about 2 MB) and the course's `ticketnet.py` into a folder `ticket-router`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: train 4696 tickets, valid 600, test 600.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `ticket-router` (in Colab: the **Files** panel on the left), then run the cell again.

The tickets are made up for this course by a language model, from a list of situations; no real customer wrote them.

In [ ]:
# Setup: download the tickets and ticketnet.py into ticket-router/ and check them.
import hashlib, os, subprocess, sys, urllib.request
from pathlib import Path

LABS = os.environ.get("NX_LABS_RAW", "https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/C06/")
FILES = {
    "train.csv": "98487169346f8b0d9bc55a3844849ab95d012b75f01d7083605a6a1cea57aa39",
    "valid.csv": "f0c7bfa75e75837cdaf891ac8d007a78a018ba8e3488155cf43f2c6a34a064c8",
    "test.csv": "f373f402bcc9742cdcf8829720710a97b7a1dce5d8e34aaa88d03e7e49a5404e",
    "split_manifest.json": "d043cf720697054fb16e6cfff72ecb49b471d75038308ad5836779a0dd0bf8a5",
}
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
PROJECT = Path("ticket-router").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:  # Python without certificates (python.org on macOS): try curl
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / "data" / name
    if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        download(LABS + "data/" + name, path)
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder ticket-router and run again.")
download(LABS + "project/ticketnet.py", PROJECT / "ticketnet.py")
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
import ticketnet
from ticketnet import *  # TEAMS, Vocab, tokenize, the three networks, train(), evaluate() ...
train_rows, valid_rows, test_rows = (load_split(f"data/{p}.csv") for p in ("train", "valid", "test"))
print(f"Ready: train {len(train_rows)} tickets, valid {len(valid_rows)}, test {len(test_rows)}.")

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.
- `show_shape(name, tensor)` prints a tensor's shape, as the lessons do.

Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it.
import hashlib, json, warnings
import numpy as np
import pandas as pd
pd.set_option("display.precision", 4)
pd.set_option("display.max_columns", 30)
warnings.filterwarnings("ignore", category=FutureWarning)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

torch.set_num_threads(2)  # the numbers in the lessons were measured on a CPU

def fingerprint_tensor(t):
    return fingerprint([round(float(v), 4) for v in t.flatten().tolist()])

def show_shape(name, t):
    print(f"{name:<12} shape {tuple(t.shape)}")

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Multi-head attention

`nn.MultiheadAttention` is PyTorch's attention layer. `num_heads=2` runs two heads side by side, each on 4 of the 8 numbers.

> **Predict.** What is the shape of the attention weights for 1 ticket, 2 heads and 3 tokens? Run the cell.

In [ ]:
from torch import nn

torch.manual_seed(0)
x = torch.randn(1, 3, 8)              # 1 ticket, 3 tokens, 8 numbers per token
attention = nn.MultiheadAttention(embed_dim=8, num_heads=2, batch_first=True)
out, weights = attention(x, x, x, average_attn_weights=False)
print("input  ", tuple(x.shape))
print("output ", tuple(out.shape))
print("weights", tuple(weights.shape))
print(weights[0].detach().numpy().round(2))

> **Check.** You should see `weights (1, 2, 3, 3)`: one 3 × 3 table for each head. The two tables differ.

More heads cut the same numbers into more slices. Run the cell.

In [ ]:
for heads in (1, 4, 8):
    layer = nn.MultiheadAttention(embed_dim=64, num_heads=heads, batch_first=True)
    print(heads, "heads:", sum(p.numel() for p in layer.parameters()), "parameters")

> **Check.** You should see 16640 parameters three times.

## 2. Feed-forward, residual connections and layer normalization

The feed-forward layer works on each token alone. Run the cell.

In [ ]:
torch.manual_seed(0)
feed_forward = nn.Sequential(nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 64))
h = torch.randn(1, 5, 64)                       # 1 ticket, 5 tokens
together = feed_forward(h)                      # all 5 tokens at once
alone = feed_forward(h[:, 2])                   # token 2 on its own
print(tuple(together.shape), tuple(alone.shape))
print(torch.allclose(together[:, 2], alone, atol=1e-6))
print(sum(p.numel() for p in feed_forward.parameters()), "parameters")

> **Check.** You should see `True` and `33088 parameters`.

> **Predict.** A stack of 20 sigmoid layers: how large is the first layer's gradient without residual connections, and with them? Run the cell.

In [ ]:
def first_layer_gradient(residual, depth=20):
    torch.manual_seed(0)
    layers = [nn.Sequential(nn.Linear(64, 64), nn.Sigmoid()) for _ in range(depth)]
    h = torch.randn(32, 64)
    for layer in layers:
        h = h + layer(h) if residual else layer(h)   # keep h and add the change, or replace h
    h.pow(2).mean().backward()
    return layers[0][0].weight.grad.norm().item()

print(f"without residual connections: {first_layer_gradient(False):.2e}")
print(f"with residual connections:    {first_layer_gradient(True):.2e}")

> **Check.** You should see `3.44e-18` without residual connections and `9.55e-01` with them.

Layer normalization gives each token's numbers mean 0 and spread 1. Run the cell.

In [ ]:
row = torch.tensor([[2.0, 4.0, 6.0, 8.0]])
print(nn.LayerNorm(4)(row).detach())
print(nn.LayerNorm(4)(row * 1000).detach())

> **Check.** You should see `-1.3416, -0.4472, 0.4472, 1.3416` twice.

## 3. One block

`nn.TransformerEncoderLayer` is the whole block: attention, feed-forward, two residual connections and two LayerNorms. Run the cell.

In [ ]:
torch.manual_seed(0)
block = nn.TransformerEncoderLayer(d_model=64, nhead=4, dim_feedforward=256, dropout=0.1,
                                   batch_first=True, norm_first=True)
x = torch.randn(8, 120, 64)             # 8 tickets, 120 tokens, 64 numbers per token
print(tuple(block(x).shape))

> **Check.** You should see `(8, 120, 64)`: the same shape in and out.

> **Your turn.** Put the number of parameters of `block` in `block_parameters`. Predict it first from the parts. Then run the check cell.

In [ ]:
block_parameters = ...

In [ ]:
expect_hash('block_parameters', block_parameters, 'a966ed2bc7c3e5f4')

## 4. The tiny transformer on the tickets

The next cell builds the vocabulary and the padded token IDs, and prints the shape after each step of `TinyTransformer` for 8 validation tickets. Run the cell.

In [ ]:
vocab = Vocab([row["text"] for row in train_rows])
ids_train, ids_valid = token_ids(train_rows, vocab), token_ids(valid_rows, vocab)
y_train, y_valid = labels(train_rows), labels(valid_rows)

set_seed(0)
model = TinyTransformer(len(vocab))
model.eval()
ids = token_ids(valid_rows[:8], vocab)
with torch.no_grad():
    h = model.embed(ids) + model.position(torch.arange(ids.shape[1]))
    show_shape("tokens", h)
    h = model.blocks(h, src_key_padding_mask=ids == PAD)
    show_shape("2 blocks", h)
    print("scores", tuple(model(ids).shape))
print("parameters", count_parameters(model))

> **Check.** You should see (8, 93, 64) twice, the scores (8, 5) and `parameters 291397`.

> **Warning.** The next cell trains for about one to three minutes. The lesson trains for 20 epochs; here it is 8, so the numbers are lower than on the lesson page. Run the cell.

In [ ]:
set_seed(0)
model = TinyTransformer(len(vocab))
history = train(model, ids_train, y_train, ids_valid, y_valid, epochs=8, lr=1e-3)

> **Check.** The validation accuracy should be about 0.92 after epoch 8. The bag-of-words network got about 0.98 in Module 3.

Accuracy by writing style. Run the cell.

In [ ]:
from collections import Counter

model.eval()
with torch.no_grad():
    pred = model(ids_valid).argmax(1)
right, total = Counter(), Counter()
for p, row in zip(pred.tolist(), valid_rows):
    total[row["style"]] += 1
    right[row["style"]] += TEAMS[p] == row["team"]
for style, n in total.most_common():
    print(f"{style:<13} {n:>3} tickets  {right[style] / n:6.1%}")

> **Check.** `long` and `two_topics` should be among the lowest lines.

## 5. Change one thing: eight heads

> **Your turn.** Build `eight = TinyTransformer(len(vocab), heads=8)` with seed 0, and put its number of parameters in `eight_parameters`. Predict it first. Then run the check cell.

In [ ]:
eight_parameters = ...

In [ ]:
expect_hash('eight_parameters', eight_parameters, 'db17155cbcb54344')

## 6. Failure case: the batch in the wrong place

Tomás forgets `batch_first=True`. The cell changes the other 7 tickets of a batch and checks whether ticket 0's output changes.

> **Predict.** Should ticket 0's output depend on the other tickets? Run the cell.

In [ ]:
torch.manual_seed(0)
wrong = nn.TransformerEncoderLayer(64, 4, 256, dropout=0.0)          # batch_first is False
right_layer = nn.TransformerEncoderLayer(64, 4, 256, dropout=0.0, batch_first=True)
wrong.eval(); right_layer.eval()
batch = torch.randn(8, 10, 64)
other = batch.clone()
other[1:] = torch.randn(7, 10, 64)               # the same ticket 0, seven other tickets
with torch.no_grad():
    print("wrong: ticket 0 changed:", not torch.allclose(wrong(batch)[0], wrong(other)[0], atol=1e-6))
    print("right: ticket 0 changed:", not torch.allclose(right_layer(batch)[0], right_layer(other)[0], atol=1e-6))

> **Check.** You should see `True` for the wrong layer and `False` for the right one. Without `batch_first=True`, attention mixes the tickets of a batch.

## Next

You built a transformer block, trained the tiny transformer, and saw that it is Larkfield's weakest router. The [lesson page](https://learning.nextia-ai.com/courses/deep-learning/m06/transformer-blocks/) has the block diagram, the honest comparison and the knowledge checks. Next, you give the model word positions and control what each token may read.

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Position and masking](https://learning.nextia-ai.com/courses/deep-learning/m06/position-and-masking/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/deep-learning/m06/transformer-blocks/)

<img src="https://raw.githubusercontent.com/usmanahmed99/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.